In [1]:
!pip install torchao --upgrade -q
!pip install transformers peft datasets evaluate scikit-learn accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [1]:
import torch
import numpy as np
import json
import time
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from peft import AdaLoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import matthews_corrcoef
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
dataset = load_dataset("glue", "cola")

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-base",
    use_fast=False
)

def tokenize(examples):
    return tokenizer(
        examples['sentence'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_loader = DataLoader(tokenized['train'], batch_size=16, shuffle=True)
val_loader = DataLoader(tokenized['validation'], batch_size=16)

# Class weights — same as LoRA notebook
labels_all = np.array(tokenized['train']['labels'])
class_counts = np.bincount(labels_all)
total = len(labels_all)
class_weights = torch.tensor(
    [total / (2 * class_counts[0]),
     total / (2 * class_counts[1])],
    dtype=torch.float32
).to(device)
print("Class weights:", class_weights)
print("Train size:", len(tokenized['train']))
print("Val size:", len(tokenized['validation']))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

Class weights: tensor([1.6913, 0.7099], device='cuda:0')
Train size: 8551
Val size: 1043


In [3]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=2,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32
).to(device)

# Quick forward pass check
base_model.eval()
sample_ids = tokenized['train'][:2]['input_ids'].to(device)
sample_mask = tokenized['train'][:2]['attention_mask'].to(device)
sample_labels = tokenized['train'][:2]['labels'].to(device)

with torch.no_grad():
    test_out = base_model(input_ids=sample_ids, attention_mask=sample_mask, labels=sample_labels)

print("Test loss:", test_out.loss.item())
print("Any NaN?", torch.isnan(test_out.logits).any().item())
# Must see: real loss number + False

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias        

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Test loss: 0.6897065043449402
Any NaN? False


In [5]:
# Cell 5 — AdaLoRA config + apply (fixed for PEFT 0.19.1)

# Calculate total steps first — AdaLoRA needs this upfront
n_epochs = 8
total_steps = len(train_loader) * n_epochs  # 535 * 8 = 4280
warmup_steps = int(0.1 * total_steps)       # 428

adalora_config = AdaLoraConfig(
    task_type=TaskType.SEQ_CLS,
    init_r=12,
    target_r=8,
    beta1=0.85,
    beta2=0.85,
    tinit=200,
    tfinal=1000,
    deltaT=10,
    lora_alpha=32,
    target_modules=["query_proj", "key_proj", "value_proj", "out_proj"],
    lora_dropout=0.1,
    total_step=total_steps      # REQUIRED in PEFT 0.19.1
)

model_ada = get_peft_model(base_model, adalora_config).to(device)
model_ada.print_trainable_parameters()

# Verify clean start
model_ada.eval()
with torch.no_grad():
    test_out = model_ada(
        input_ids=sample_ids,
        attention_mask=sample_mask,
        labels=sample_labels
    )
print("Post-AdaLoRA test loss:", test_out.loss.item())
print("Any NaN?", torch.isnan(test_out.logits).any().item())

trainable params: 665,522 || all params: 185,089,240 || trainable%: 0.3596
Post-AdaLoRA test loss: 1.8913710117340088
Any NaN? False


In [6]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model_ada.parameters()),
    lr=1e-4,
    weight_decay=0.01,
    eps=1e-6
)

n_epochs = 8
total_steps = len(train_loader) * n_epochs
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

results_ada = {
    "epoch": [], "train_loss": [],
    "val_mcc": [], "epoch_time_sec": []
}

global_step = 0  # AdaLoRA needs this to track when to reallocate rank

for epoch in range(n_epochs):
    model_ada.train()
    total_loss = 0
    valid_batches = 0
    nan_count = 0
    start_time = time.time()

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model_ada(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        loss = loss_fn(outputs.logits, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            global_step += 1
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ada.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        # CRITICAL — this is what makes AdaLoRA actually adaptive
        # Updates importance scores and reallocates rank across layers
        model_ada.update_and_allocate(global_step)

        global_step += 1
        total_loss += loss.item()
        valid_batches += 1

    # Validation
    model_ada.eval()
    all_preds, all_labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            outputs = model_ada(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(batch['labels'].numpy())

    mcc = matthews_corrcoef(all_labels_list, all_preds)
    epoch_time = time.time() - start_time
    avg_loss = total_loss / valid_batches if valid_batches > 0 else float('nan')

    results_ada["epoch"].append(epoch + 1)
    results_ada["train_loss"].append(round(avg_loss, 4))
    results_ada["val_mcc"].append(round(mcc, 4))
    results_ada["epoch_time_sec"].append(round(epoch_time, 1))

    print(f"Epoch {epoch+1}/{n_epochs} | Loss: {avg_loss:.4f} | MCC: {mcc:.4f} | "
          f"Time: {epoch_time:.1f}s | NaN: {nan_count} | Step: {global_step}")

# Save immediately
with open('/content/adalora_results.json', 'w') as f:
    json.dump(results_ada, f, default=str)

print(f"\n=== ADALORA RESULTS ===")
print(f"Best MCC: {max(results_ada['val_mcc']):.4f}")
print(f"Avg epoch time: {np.mean(results_ada['epoch_time_sec']):.1f}s")
print(results_ada)

Epoch 1/8 | Loss: 0.6944 | MCC: 0.0377 | Time: 195.6s | NaN: 0 | Step: 535
Epoch 2/8 | Loss: 0.6951 | MCC: 0.0544 | Time: 188.6s | NaN: 0 | Step: 1070
Epoch 3/8 | Loss: 0.6952 | MCC: 0.0246 | Time: 188.5s | NaN: 0 | Step: 1605
Epoch 4/8 | Loss: 0.6955 | MCC: 0.0622 | Time: 188.8s | NaN: 0 | Step: 2140
Epoch 5/8 | Loss: 0.6939 | MCC: 0.0901 | Time: 188.4s | NaN: 0 | Step: 2675
Epoch 6/8 | Loss: 0.6919 | MCC: 0.0862 | Time: 188.0s | NaN: 0 | Step: 3210
Epoch 7/8 | Loss: 0.6917 | MCC: 0.0857 | Time: 188.6s | NaN: 0 | Step: 3745
Epoch 8/8 | Loss: 0.6936 | MCC: 0.0886 | Time: 188.9s | NaN: 0 | Step: 4280

=== ADALORA RESULTS ===
Best MCC: 0.0901
Avg epoch time: 189.4s
{'epoch': [1, 2, 3, 4, 5, 6, 7, 8], 'train_loss': [0.6944, 0.6951, 0.6952, 0.6955, 0.6939, 0.6919, 0.6917, 0.6936], 'val_mcc': [np.float64(0.0377), np.float64(0.0544), np.float64(0.0246), np.float64(0.0622), np.float64(0.0901), np.float64(0.0862), np.float64(0.0857), np.float64(0.0886)], 'epoch_time_sec': [195.6, 188.6, 188.5,

In [9]:
# Cell 7 Final Fix

def compute_effective_rank(matrix):
    with torch.no_grad():
        matrix_cpu = matrix.float().cpu()
        try:
            _, sv, _ = torch.linalg.svd(matrix_cpu, full_matrices=False)
        except Exception:
            _, sv_np, _ = np.linalg.svd(matrix_cpu.numpy(), full_matrices=False)
            sv = torch.tensor(sv_np)
        sv = sv.float()
        sv = sv[sv > 1e-10]
        if len(sv) == 0:
            return 0.0
        p = sv / sv.sum()
        return torch.exp(-torch.sum(p * torch.log(p))).item()

print("=== AdaLoRA Per-Layer Rank Allocation ===")
eranks_ada = []
layer_info = []

for name, module in model_ada.named_modules():
    if hasattr(module, 'lora_A') and hasattr(module, 'lora_B') and hasattr(module, 'lora_E'):
        for key in module.lora_A.keys():
            A = module.lora_A[key]          # shape [12, 768]
            B = module.lora_B[key]          # shape [768, 12]
            E = module.lora_E[key].squeeze() # shape [12, 1] → [12]

            combined = B @ torch.diag(E) @ A  # [768,12] @ [12,12] @ [12,768] = [768,768]
            erank = compute_effective_rank(combined)
            active_rank = (E.abs() > 1e-6).sum().item()
            eranks_ada.append(erank)
            layer_info.append({
                "name": name,
                "active_rank": int(active_rank),
                "erank": round(erank, 3)
            })
            print(f"Layer {len(eranks_ada):02d} | {name.split('.')[-1]} "
                  f"| Active rank: {active_rank}/{adalora_config.init_r} "
                  f"| Eff rank: {erank:.3f}")

print(f"\n=== SUMMARY ===")
print(f"Mean effective rank: {np.mean(eranks_ada):.3f}")
print(f"Min effective rank:  {np.min(eranks_ada):.3f}")
print(f"Max effective rank:  {np.max(eranks_ada):.3f}")
print(f"Init rank: {adalora_config.init_r} | Target rank: {adalora_config.target_r}")
print(f"Layers with full active rank ({adalora_config.init_r}): "
      f"{sum(1 for l in layer_info if l['active_rank'] == adalora_config.init_r)}")
print(f"Layers with pruned rank (<{adalora_config.init_r}): "
      f"{sum(1 for l in layer_info if l['active_rank'] < adalora_config.init_r)}")

ada_summary = {
    "mean_erank": round(float(np.mean(eranks_ada)), 3),
    "min_erank": round(float(np.min(eranks_ada)), 3),
    "max_erank": round(float(np.max(eranks_ada)), 3),
    "per_layer": layer_info
}
with open('/content/adalora_erank.json', 'w') as f:
    json.dump(ada_summary, f)
print("Saved.")

=== AdaLoRA Per-Layer Rank Allocation ===
Layer 01 | query_proj | Active rank: 10/12 | Eff rank: 8.171
Layer 02 | key_proj | Active rank: 12/12 | Eff rank: 9.419
Layer 03 | value_proj | Active rank: 12/12 | Eff rank: 8.074
Layer 04 | query_proj | Active rank: 11/12 | Eff rank: 9.987
Layer 05 | key_proj | Active rank: 11/12 | Eff rank: 9.155
Layer 06 | value_proj | Active rank: 11/12 | Eff rank: 6.521
Layer 07 | query_proj | Active rank: 8/12 | Eff rank: 6.695
Layer 08 | key_proj | Active rank: 11/12 | Eff rank: 9.530
Layer 09 | value_proj | Active rank: 11/12 | Eff rank: 8.388
Layer 10 | query_proj | Active rank: 12/12 | Eff rank: 9.352
Layer 11 | key_proj | Active rank: 11/12 | Eff rank: 8.637
Layer 12 | value_proj | Active rank: 12/12 | Eff rank: 9.488
Layer 13 | query_proj | Active rank: 7/12 | Eff rank: 5.705
Layer 14 | key_proj | Active rank: 7/12 | Eff rank: 5.731
Layer 15 | value_proj | Active rank: 12/12 | Eff rank: 7.825
Layer 16 | query_proj | Active rank: 6/12 | Eff rank: 5.